# 02. Political Text Preprocessing Pipeline

- **Role/Owner:** Member 2 (ML & Data Engineering Lead)
- **Core Responsibility:** Clean and normalize political claim text (HTML entity decoding, Unicode NFKD + homoglyph folding, emoji stripping, URL / @mention masking, camelCase hashtag splitting, sensational punctuation collapsing), then build the stratified train/val/test splits.
- **Key Interface/Contract:** Consumes the raw PolitiFact CSVs and PHEME cascade pickle; outputs unified train/val/test CSVs to `backend/data/processed/` and mirrors its cleaning logic in `app/services/preprocessor.py`.

In [1]:
# Environment & Dependency Setup
import re
import html
import time
import pickle
import unicodedata
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)

print('02_preprocessing dependencies imported successfully.')

02_preprocessing dependencies imported successfully.


## Part 1: Raw Corpus Loading
We rebuild the unified raw corpus exactly as audited in `01_data_exploration.ipynb`: PolitiFact claim titles (432 fake, 624 real) plus the root/seed tweet of each of the 6,425 PHEME cascades. Only seed texts enter the NLP corpus; reply trees stay in `cascades` for graph features.

In [2]:
# Load PolitiFact claims
fake = pd.read_csv('../data/fakenewsnet_politifact/politifact_fake.csv')
real = pd.read_csv('../data/fakenewsnet_politifact/politifact_real.csv')

politifact_df = pd.concat([
    fake.assign(label=1),
    real.assign(label=0),
], ignore_index=True)
politifact_df = pd.DataFrame({
    'id': politifact_df['id'].astype(str),
    'text': politifact_df['title'],
    'label': politifact_df['label'],
    'source': 'politifact',
    'event': 'politifact',
})

# Load PHEME cascades and extract root/seed tweets
with open('../data/pheme_political/pheme_cascades.gpickle', 'rb') as f:
    cascades = pickle.load(f)

pheme_rows = []
for g in cascades:
    thread_id = str(g.graph.get('thread_id'))
    root = g.nodes[thread_id] if thread_id in g.nodes else {}
    pheme_rows.append({
        'id': thread_id,
        'text': str(root.get('text', '')),
        'label': int(g.graph.get('label')),
        'source': 'pheme',
        'event': str(g.graph.get('event')),
    })
pheme_df = pd.DataFrame(pheme_rows)

raw_df = pd.concat([politifact_df, pheme_df], ignore_index=True)
assert len(raw_df) == 7481, f'Expected 7,481 rows, found {len(raw_df)}'

print(f'Raw corpus: {len(raw_df):,} records | PolitiFact: {len(politifact_df):,} | PHEME: {len(pheme_df):,}')
print(f'Misinformation share: {raw_df["label"].mean():.1%}')

Raw corpus: 7,481 records | PolitiFact: 1,056 | PHEME: 6,425
Misinformation share: 37.9%


## Part 2: Cleaning Function Implementation
Each stage is a small pure function so it can be unit-tested and copied verbatim into the FastAPI service. Stage order matters:

1. **HTML entities** (`&amp;` → `&`) — 322 PHEME tweets carry escaped entities.
2. **Unicode normalization** — folds common Cyrillic/Greek homoglyphs to Latin, applies NFKD (fullwidth letters, ligatures, `…` → `...`), then drops combining accents (`café` → `cafe`). NFKD alone does *not* map Cyrillic `а` to Latin `a`, hence the explicit homoglyph table.
3. **Emoji stripping** — emoji, pictographs, dingbats, flags, ZWJ and variation selectors.
4. **Entity masking** — URLs → `[URL]`, handles → `@USER`. Runs before hashtag splitting so `#` fragments inside URLs are never touched.
5. **Hashtag splitting** — `#StopTheSteal` → `Stop The Steal`, `#BBCNews` → `BBC News`, `#COVID19` → `COVID 19`; lowercase tags just lose the `#`.
6. **Punctuation** — runs of `!`/`?` collapse to one mark set off as its own token (`Fraud!!!!!` → `Fraud !`), so the sensationalism signal survives as a consistent token; `....` → `.`.
7. **Whitespace collapse and word truncation** at `max_length` words.

In [3]:
# Cyrillic and Greek characters that render identically to Latin letters
HOMOGLYPHS = str.maketrans({
    'а': 'a', 'е': 'e', 'о': 'o', 'р': 'p', 'с': 'c', 'у': 'y', 'х': 'x',
    'і': 'i', 'ј': 'j', 'ѕ': 's', 'һ': 'h', 'ԁ': 'd', 'ɡ': 'g',
    'А': 'A', 'В': 'B', 'Е': 'E', 'К': 'K', 'М': 'M', 'Н': 'H', 'О': 'O',
    'Р': 'P', 'С': 'C', 'Т': 'T', 'Х': 'X', 'І': 'I', 'Ј': 'J', 'Ѕ': 'S',
    'ο': 'o', 'α': 'a', 'ν': 'v', 'ι': 'i',
    'Α': 'A', 'Β': 'B', 'Ε': 'E', 'Ζ': 'Z', 'Η': 'H', 'Ι': 'I', 'Κ': 'K',
    'Μ': 'M', 'Ν': 'N', 'Ο': 'O', 'Ρ': 'P', 'Τ': 'T', 'Υ': 'Y', 'Χ': 'X',
})

EMOJI_PATTERN = re.compile(
    '['
    '\U0001F000-\U0001FAFF'  # emoticons, pictographs, transport, flags, supplemental symbols
    '☀-➿'          # misc symbols and dingbats
    '⬀-⯿'          # arrows and stars used as decoration
    '︎️‍'     # variation selectors and zero-width joiner
    ']+'
)
URL_PATTERN = re.compile(r'(?:https?://|www\.)\S+', flags=re.IGNORECASE)
MENTION_PATTERN = re.compile(r'(?<!\w)@\w+')
HASHTAG_PATTERN = re.compile(r'#+(\w+)')


def normalize_unicode(text: str) -> str:
    """Fold homoglyphs, apply NFKD, and drop combining accent marks."""
    text = text.translate(HOMOGLYPHS)
    text = unicodedata.normalize('NFKD', text)
    return ''.join(ch for ch in text if not unicodedata.combining(ch))


def remove_emojis(text: str) -> str:
    """Strip emoji and decorative pictographic symbols."""
    return EMOJI_PATTERN.sub(' ', text)


def mask_entities(text: str) -> str:
    """Replace URLs with [URL] and Twitter handles with @USER."""
    text = URL_PATTERN.sub('[URL]', text)
    return MENTION_PATTERN.sub('@USER', text)


def _split_tag(match: re.Match) -> str:
    tag = match.group(1).replace('_', ' ')
    tag = re.sub(r'([a-z])([A-Z])', r'\1 \2', tag)        # StopThe -> Stop The
    tag = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1 \2', tag)  # BBCNews -> BBC News
    tag = re.sub(r'([A-Za-z])(\d)', r'\1 \2', tag)        # COVID19 -> COVID 19
    return tag


def split_hashtags(text: str) -> str:
    """Expand #CamelCase hashtags into words and drop the # symbol."""
    return HASHTAG_PATTERN.sub(_split_tag, text)


def normalize_punctuation(text: str) -> str:
    """Collapse repeated !/? into one standalone mark and repeated dots into one."""
    text = re.sub(r'\s*([!?])[!?]*', r' \1', text)
    return re.sub(r'\.{2,}', '.', text)


def clean_political_text(text, max_length: int = 512) -> str:
    """Run the full cleaning pipeline; None, NaN and blank input return ''."""
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ''
    text = str(text)
    if not text.strip():
        return ''

    cleaned = html.unescape(text)
    cleaned = normalize_unicode(cleaned)
    cleaned = remove_emojis(cleaned)
    cleaned = mask_entities(cleaned)
    cleaned = split_hashtags(cleaned)
    cleaned = normalize_punctuation(cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    words = cleaned.split()
    if len(words) > max_length:
        cleaned = ' '.join(words[:max_length])
    return cleaned


print('Cleaning pipeline defined: normalize_unicode, remove_emojis, mask_entities, '
      'split_hashtags, normalize_punctuation, clean_political_text')

Cleaning pipeline defined: normalize_unicode, remove_emojis, mask_entities, split_hashtags, normalize_punctuation, clean_political_text


## Part 3: Edge-Case Unit Assertion Suite
Every stage is checked in isolation and through the full pipeline, including the empty, `None`, NaN and purely symbolic inputs that must return a string without raising.

In [4]:
cases = [
    # (input, expected, description)
    ('#StopTheSteal', 'Stop The Steal', 'CamelCase hashtag'),
    ('#FergusonDecision', 'Ferguson Decision', 'PascalCase hashtag'),
    ('#BBCNews live', 'BBC News live', 'acronym hashtag'),
    ('#COVID19', 'COVID 19', 'alphanumeric hashtag'),
    ('#ferguson', 'ferguson', 'lowercase hashtag'),
    ('##ParisShooting', 'Paris Shooting', 'doubled hash typo'),
    ('#black_lives_matter', 'black lives matter', 'underscore hashtag'),
    ('Breaking: https://t.co/xyz123', 'Breaking: [URL]', 'https URL'),
    ('see http://a.co/1 and www.b.com/x', 'see [URL] and [URL]', 'multiple URLs'),
    ('@realDonaldTrump says hello', '@USER says hello', 'mention'),
    ('RT .@rolandsmartin: wow', 'RT .@USER: wow', 'retweet mention'),
    ('mail me at a@b.com', 'mail me at a@b.com', 'email is not a mention'),
    ('Fraud!!!!!', 'Fraud !', 'repeated exclamation'),
    ('Really???', 'Really ?', 'repeated question'),
    ('What?!?!', 'What ?', 'mixed !? run'),
    ('Wait....', 'Wait.', 'repeated dots'),
    ('Wait…', 'Wait.', 'unicode ellipsis'),
    ('Price is 3.5%', 'Price is 3.5%', 'decimals untouched'),
    ('Cops &amp; protesters', 'Cops & protesters', 'HTML entity'),
    ('Hard to believe 😂😂😂', 'Hard to believe', 'emoji'),
    ('Flag 🇺🇸 here ❤️', 'Flag here', 'flag and heart emoji'),
    ('Тrump', 'Trump', 'Cyrillic T homoglyph'),
    ('Оbаmа', 'Obama', 'Cyrillic O/a homoglyphs'),
    ('café naïve', 'cafe naive', 'accents stripped'),
    ('ＦＡＫＥ ﬁle', 'FAKE file', 'fullwidth letters and ligature'),
    ('', '', 'empty string'),
    ('   \n\t ', '', 'whitespace only'),
    (None, '', 'None input'),
    (float('nan'), '', 'NaN input'),
    ('😂😂 🔥', '', 'purely symbolic input'),
    ('  multiple   spaces\nand\tnewlines ', 'multiple spaces and newlines', 'whitespace collapse'),
]

failures = []
for raw, expected, desc in cases:
    got = clean_political_text(raw)
    if got != expected:
        failures.append((desc, raw, expected, got))

# Truncation keeps exactly max_length words
long_text = ' '.join(f'w{i}' for i in range(1000))
assert len(clean_political_text(long_text).split()) == 512
assert len(clean_political_text(long_text, max_length=10).split()) == 10
assert clean_political_text('short text', max_length=10) == 'short text'

# Stage-level checks
assert split_hashtags('#StopTheSteal') == 'Stop The Steal'
assert mask_entities('@CNN at https://cnn.com') == '@USER at [URL]'
assert normalize_punctuation('No!!!') == 'No !'
assert remove_emojis('a😀b').split() == ['a', 'b']
assert normalize_unicode('Тrump') == 'Trump'

# Idempotence: cleaning cleaned text changes nothing
for raw, _, _ in cases:
    once = clean_political_text(raw)
    assert clean_political_text(once) == once, f'Not idempotent: {raw!r}'

for desc, raw, expected, got in failures:
    print(f'FAIL [{desc}] {raw!r}: expected {expected!r}, got {got!r}')
assert not failures, f'{len(failures)} assertion(s) failed'
print(f'All {len(cases)} pipeline cases + truncation, stage-level and idempotence checks passed (0 failures).')

All 31 pipeline cases + truncation, stage-level and idempotence checks passed (0 failures).


## Part 4: Before/After Inspection on the Real Corpus
We run the pipeline on the full raw corpus once to see which artifacts it removes and to confirm no record becomes empty. (Phase 4 persists these results into the splits.)

In [5]:
preview_df = raw_df.copy()
preview_df['text_clean'] = preview_df['text'].apply(clean_political_text)

artifacts = {
    'URLs': r'(?:https?://|www\.)\S+',
    '@mentions (raw handles)': r'(?<!\w)@(?!USER\b)\w+',
    'hashtags (#)': r'#\w+',
    'HTML entities': r'&(?:amp|lt|gt|quot|#\d+);',
    'emoji': EMOJI_PATTERN.pattern,
    'repeated !/?': r'[!?]{2,}',
    'repeated dots / …': r'\.{2,}|…',
}
artifact_report = pd.DataFrame({
    'records_before': {k: preview_df['text'].str.contains(p, regex=True).sum() for k, p in artifacts.items()},
    'records_after': {k: preview_df['text_clean'].str.contains(p, regex=True).sum() for k, p in artifacts.items()},
})
display(artifact_report)

empty_after = (preview_df['text_clean'] == '').sum()
print(f'Records empty after cleaning: {empty_after}')
print(f'Mean words  raw: {preview_df["text"].str.split().str.len().mean():.1f} | '
      f'clean: {preview_df["text_clean"].str.split().str.len().mean():.1f}')

changed = preview_df[preview_df['text'] != preview_df['text_clean']]
print(f'Records changed by cleaning: {len(changed):,} / {len(preview_df):,}')
display(changed.sample(8, random_state=42)[['source', 'label', 'text', 'text_clean']])

,records_before,records_after
URLs,4210,0
@mentions (raw handles),976,0
hashtags (#),4615,0
HTML entities,323,0
emoji,64,0
repeated !/?,64,0
repeated dots / …,261,0


Records empty after cleaning: 0
Mean words  raw: 15.2 | clean: 15.8
Records changed by cleaning: 6,093 / 7,481


,source,label,text,text_clean
4542,pheme,0,The most eloquent responses to today's tragedy have come from the cartoonists. #JeSuisCharlie RT @MaximeHaes\n http:...,The most eloquent responses to today's tragedy have come from the cartoonists. Je Suis Charlie RT @USER [URL]
3155,pheme,0,The plot thickens: #Gurlitt's cousin claims Munich #art trove as inheritance http://t.co/Z8k55lzPLp via @YahooNews,The plot thickens: Gurlitt's cousin claims Munich art trove as inheritance [URL] via @USER
4605,pheme,0,"If you're a media outlet reporting on #CharlieHebdo and blurring out the cartoons you're reporting on, you're doing ...","If you're a media outlet reporting on Charlie Hebdo and blurring out the cartoons you're reporting on, you're doing ..."
1531,pheme,0,RT @YourAnonNews: SWAT police just entered a McDonald's in #Ferguson and started asking journalists for ID http://t....,RT @USER: SWAT police just entered a McDonald's in Ferguson and started asking journalists for ID [URL] (via.
3750,pheme,0,NEWS Germanwings flight crashes in south of France http://t.co/Iy99ucFqQ0,NEWS Germanwings flight crashes in south of France [URL]
4285,pheme,1,Gunman at Paris grocery store threatens to kill hostages if police storm the #CharlieHebdo brothers. http://t.co/EBb...,Gunman at Paris grocery store threatens to kill hostages if police storm the Charlie Hebdo brothers. [URL]
1233,pheme,1,"the police version is also that #MikeBrown approached the officer. But if he had JUST robbed a store, why approach p...","the police version is also that Mike Brown approached the officer. But if he had JUST robbed a store, why approach p..."
2014,pheme,0,"The courage it takes to to write these words down, much less display them to the world #Ferguson #MikeBrown http://t...","The courage it takes to to write these words down, much less display them to the world Ferguson Mike Brown [URL]"


## Part 5: Throughput Benchmark
The plan requires `clean_political_text` to process a 1,000-text sample in under 2 seconds, so it stays cheap enough for the FastAPI request path.

In [6]:
sample_texts = raw_df['text'].sample(1000, random_state=42).tolist()

timings = []
for _ in range(5):
    start = time.perf_counter()
    _ = [clean_political_text(t) for t in sample_texts]
    timings.append(time.perf_counter() - start)

best, mean = min(timings), sum(timings) / len(timings)
print(f'1,000 texts: best {best * 1000:.1f} ms | mean {mean * 1000:.1f} ms '
      f'| {1000 / mean:,.0f} texts/sec')
assert mean < 2.0, f'Benchmark too slow: {mean:.2f}s for 1,000 texts'
print('Throughput requirement met (< 2 s per 1,000 texts).')

1,000 texts: best 15.6 ms | mean 15.9 ms | 62,904 texts/sec
Throughput requirement met (< 2 s per 1,000 texts).
